# Final Project

This notebook is adapted from here: https://aiqm.github.io/torchani/examples/nnp_training.html

## Checkpoint 1: Data preparation

1. Create a working directory: `/global/scratch/users/[USER_NAME]/[DIR_NAME]`. Replace the [USER_NAME] with yours and specify a [DIR_NAME] you like.
2. Copy the Jupyter Notebook to the working directory
3. Download the ANI dataset `ani_dataset_gdb_s01_to_s04.h5` from bCourses and upload it to the working directory

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torchani
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import random
import copy

### Use GPU

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### Set up AEV computer

#### AEV: Atomic Environment Vector (atomic features)

Ref: Chem. Sci., 2017, 8, 3192

In [3]:
def init_aev_computer():
    Rcr = 5.2
    Rca = 3.5
    EtaR = torch.tensor([16], dtype=torch.float, device=device)
    ShfR = torch.tensor([
        0.900000, 1.168750, 1.437500, 1.706250, 
        1.975000, 2.243750, 2.512500, 2.781250, 
        3.050000, 3.318750, 3.587500, 3.856250, 
        4.125000, 4.393750, 4.662500, 4.931250
    ], dtype=torch.float, device=device)


    EtaA = torch.tensor([8], dtype=torch.float, device=device)
    Zeta = torch.tensor([32], dtype=torch.float, device=device)
    ShfA = torch.tensor([0.90, 1.55, 2.20, 2.85], dtype=torch.float, device=device)
    ShfZ = torch.tensor([
        0.19634954, 0.58904862, 0.9817477, 1.37444680, 
        1.76714590, 2.15984490, 2.5525440, 2.94524300
    ], dtype=torch.float, device=device)

    num_species = 4
    aev_computer = torchani.AEVComputer(
        Rcr, Rca, EtaR, ShfR, EtaA, Zeta, ShfA, ShfZ, num_species
    )
    return aev_computer

aev_computer = init_aev_computer()
aev_dim = aev_computer.aev_length
print(aev_dim)

384


In [4]:
def load_ani_dataset(dspath):
    species_order = ['H', 'C', 'N', 'O']

    dataset = torchani.data.load_ani_dataset(
        dspath,
        torchani.utils.ChemicalSymbolsToInts(species_order),
        8192
    )
    return dataset

### Prepare dataset & split

In [5]:
def load_ani_dataset(dspath):
    self_energies = torch.tensor([
        0.500607632585, -37.8302333826,
        -54.5680045287, -75.0362229210
    ], dtype=torch.float, device=device)
    energy_shifter = torchani.utils.EnergyShifter(None)
    species_order = ['H', 'C', 'N', 'O']
    dataset = torchani.data.load(dspath)
    dataset = dataset.subtract_self_energies(energy_shifter, species_order)
    dataset = dataset.species_to_indices(species_order)
    dataset = dataset.shuffle()
    return dataset

dataset = load_ani_dataset("./ani_gdb_s01_to_s04.h5")

dataset = load_ani_dataset("./ani_gdb_s01_to_s04.h5")

train_data, val_data, test_data = dataset.split(0.8, 0.1, None)

BATCH_SIZE = 1024
print("Caching data loaders (one-time cost)...")
train_loader = train_data.collate(BATCH_SIZE).cache()
val_loader   = val_data.collate(BATCH_SIZE).cache()
test_loader  = test_data.collate(BATCH_SIZE).cache()
print("Done.")

Caching data loaders (one-time cost)...
Done.


### Training

In [6]:
class ANITrainer:
    def __init__(self, model, batch_size, learning_rate, epoch, l2):
        self.model = model
        
        num_params = sum(item.numel() for item in model.parameters())
        print(f"{model.__class__.__name__} - Number of parameters: {num_params}")
        
        self.batch_size = batch_size
        self.optimizer = torch.optim.Adam(model.parameters(), learning_rate, weight_decay=l2)
        self.epoch = epoch
    
    def train(self, train_data, val_data, early_stop=True, draw_curve=True):
        self.model.train()
        
        # init data loader
        print("Initialize training data...")
        train_data_loader = train_data.collate(self.batch_size).cache()
        
        # definition of loss function: MSE is a good choice! 
        loss_func = nn.MSELoss()
        
        # record epoch losses
        train_loss_list = []
        val_loss_list = []
        lowest_val_loss = np.inf
        
        for i in tqdm(range(self.epoch), leave=True):
            train_epoch_loss = 0.0
            for train_data_batch in train_data_loader:
                
                # compute energies
                species = train_data_batch['species'].to(device)
                coords = train_data_batch['coordinates'].to(device)
                true_energies = train_data_batch['energies'].to(device).float()
                _, pred_energies = self.model((species, coords))
                
                # compute loss
                batch_loss = loss_func(true_energies, pred_energies)
                
                # do a step
                self.optimizer.zero_grad()
                batch_loss.backward()
                self.optimizer.step()
                
                batch_importance = true_energies.shape[0]
                train_epoch_loss += batch_loss.item() * batch_importance
            
            # use the self.evaluate to get loss on the validation set 
            val_epoch_loss = self.evaluate(val_data, draw_plot=False)
            
            # append the losses
            train_loss_list.append(train_epoch_loss)
            val_loss_list.append(val_epoch_loss)
            
            if early_stop:
                if val_epoch_loss < lowest_val_loss:
                    lowest_val_loss = val_epoch_loss
                    weights = self.model.state_dict()
        
        if draw_curve:
            fig, ax = plt.subplots(1, 1, figsize=(5, 4), constrained_layout=True)
            ax.set_yscale("log")
            # Plot train loss and validation loss
            ax.plot(range(len(train_loss_list)), train_loss_list, label='Train')
            ax.plot(range(len(val_loss_list)), val_loss_list, label='Validation')
            ax.legend()
            ax.set_xlabel("# Batch")
            ax.set_ylabel("Loss")
        
        if early_stop:
            self.model.load_state_dict(weights)
        
        return train_loss_list, val_loss_list
    
    
    def evaluate(self, data, draw_plot=False, return_mae=False):
        
        # init data loader
        data_loader = data.collate(self.batch_size).cache()
        
        # init loss function
        loss_func = nn.MSELoss()
        total_loss = 0.0
        
        if draw_plot or return_mae:
            true_energies_all = []
            pred_energies_all = []
            
        with torch.no_grad():
            for batch_data in data_loader:
                
                # compute energies
                species = batch_data['species'].to(device)
                coords = batch_data['coordinates'].to(device)
                true_energies = batch_data['energies'].to(device).float()
                _, pred_energies = self.model((species, coords))
                
                # compute loss
                batch_loss = loss_func(true_energies, pred_energies)

                batch_importance = true_energies.shape[0]
                total_loss += batch_loss.item() * batch_importance
                
                if draw_plot or return_mae:
                    true_energies_all.append(true_energies.detach().cpu().numpy().flatten())
                    pred_energies_all.append(pred_energies.detach().cpu().numpy().flatten())

        if draw_plot or return_mae:
            true_energies_all = np.concatenate(true_energies_all)
            pred_energies_all = np.concatenate(pred_energies_all)
            # Report the mean absolute error
            # The unit of energies in the dataset is hartree
            # please convert it to kcal/mol when reporting the mean absolute error
            # 1 hartree = 627.5094738898777 kcal/mol
            # MAE = mean(|true - pred|)
            hartree2kcalmol = 627.5094738898777
            mae = np.mean(np.abs(true_energies_all - pred_energies_all)) * hartree2kcalmol

        if draw_plot:
            fig, ax = plt.subplots(1, 1, figsize=(5, 4), constrained_layout=True)
            ax.scatter(true_energies_all, pred_energies_all, label=f"MAE: {mae:.2f} kcal/mol", s=2)
            ax.set_xlabel("Ground Truth")
            ax.set_ylabel("Predicted")
            xmin, xmax = ax.get_xlim()
            ymin, ymax = ax.get_ylim()
            vmin, vmax = min(xmin, ymin), max(xmax, ymax)
            ax.set_xlim(vmin, vmax)
            ax.set_ylim(vmin, vmax)
            ax.plot([vmin, vmax], [vmin, vmax], color='red')
            ax.legend()
        
        if return_mae:
            return total_loss, mae
        return total_loss

### Torchani API

In [7]:
class AtomicNet(nn.Module):
    def __init__(self, aev_dim, hidden_layers, dropout=0.0):
        super().__init__()
        layers = []
        in_dim = aev_dim
        for h_dim in hidden_layers:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.layers = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.layers(x)

def build_model(aev_computer, aev_dim, hidden_layers, dropout):
    # Initialize fresh atomic networks for each element (H, C, N, O)
    net_H = AtomicNet(aev_dim, hidden_layers, dropout)
    net_C = AtomicNet(aev_dim, hidden_layers, dropout)
    net_N = AtomicNet(aev_dim, hidden_layers, dropout)
    net_O = AtomicNet(aev_dim, hidden_layers, dropout)

    ani_net = torchani.ANIModel([net_H, net_C, net_N, net_O])
    model = nn.Sequential(
        aev_computer,
        ani_net
    ).to(device)
    return model

In [23]:

# Define hyperparameter grid
hidden_layers_list = [[128]]
learning_rates = [1e-3, 1e-4]
epochs_list = [10]
l2_list = [1e-5]
dropout_list = [0.0, 0.1]

results = []

for hidden_layers, lr, epochs, l2, dropout in itertools.product(
    hidden_layers_list, learning_rates, epochs_list, l2_list, dropout_list
):
    print(f"Training: hidden={hidden_layers}, lr={lr}, epochs={epochs}, l2={l2}, dropout={dropout}")
    
    # Re-initialize model for each combination (important: avoids unfair comparison)
    model = build_model(aev_computer, aev_dim, hidden_layers, dropout)
    trainer = ANITrainer(model, 128, lr, epochs, l2)
    
    # Train the model (no plots during grid search)
    train_losses, val_losses = trainer.train(train_data, val_data, draw_curve=False)
    
    # Compute MAE on train, validation, and test sets
    _, train_mae = trainer.evaluate(train_data, return_mae=True)
    _, val_mae = trainer.evaluate(val_data, return_mae=True)
    _, test_mae = trainer.evaluate(test_data, return_mae=True)
    
    results.append({
        'Hidden Layers': str(hidden_layers),
        'Learning Rate': lr,
        'Epochs': epochs,
        'L2 Reg': l2,
        'Dropout': dropout,
        'Train MAE (kcal/mol)': round(train_mae, 2),
        'Val MAE (kcal/mol)': round(val_mae, 2),
        'Test MAE (kcal/mol)': round(test_mae, 2),
    })
    
    print(f"  -> Train MAE: {train_mae:.2f}, Val MAE: {val_mae:.2f}, Test MAE: {test_mae:.2f}\n")

Training: hidden=[128], lr=0.001, epochs=10, l2=1e-05, dropout=0.0
Sequential - Number of parameters: 197636
Initialize training data...


100%|██████████| 10/10 [06:44<00:00, 40.43s/it]


  -> Train MAE: 2.63, Val MAE: 2.63, Test MAE: 2.63

Training: hidden=[128], lr=0.001, epochs=10, l2=1e-05, dropout=0.1
Sequential - Number of parameters: 197636
Initialize training data...


100%|██████████| 10/10 [06:49<00:00, 40.93s/it]


  -> Train MAE: 3.00, Val MAE: 3.02, Test MAE: 3.01

Training: hidden=[128], lr=0.0001, epochs=10, l2=1e-05, dropout=0.0
Sequential - Number of parameters: 197636
Initialize training data...


100%|██████████| 10/10 [07:11<00:00, 43.15s/it]


  -> Train MAE: 1.21, Val MAE: 1.21, Test MAE: 1.22

Training: hidden=[128], lr=0.0001, epochs=10, l2=1e-05, dropout=0.1
Sequential - Number of parameters: 197636
Initialize training data...


100%|██████████| 10/10 [07:27<00:00, 44.77s/it]


  -> Train MAE: 1.61, Val MAE: 1.62, Test MAE: 1.63



In [ ]:
# Define hyperparameter grid
hidden_layers_list = [[256, 128]]
learning_rates = [1e-3, 1e-4]
epochs_list = [10]
l2_list = [1e-5]
dropout_list = [0.0, 0.1]

for hidden_layers, lr, epochs, l2, dropout in itertools.product(
    hidden_layers_list, learning_rates, epochs_list, l2_list, dropout_list
):
    print(f"Training: hidden={hidden_layers}, lr={lr}, epochs={epochs}, l2={l2}, dropout={dropout}")
    
    # Re-initialize model for each combination (important: avoids unfair comparison)
    model = build_model(aev_computer, aev_dim, hidden_layers, dropout)
    trainer = ANITrainer(model, 128, lr, epochs, l2)
    
    # Train the model (no plots during grid search)
    train_losses, val_losses = trainer.train(train_data, val_data, draw_curve=False)
    
    # Compute MAE on train, validation, and test sets
    _, train_mae = trainer.evaluate(train_data, return_mae=True)
    _, val_mae = trainer.evaluate(val_data, return_mae=True)
    _, test_mae = trainer.evaluate(test_data, return_mae=True)
    
    results.append({
        'Hidden Layers': str(hidden_layers),
        'Learning Rate': lr,
        'Epochs': epochs,
        'L2 Reg': l2,
        'Dropout': dropout,
        'Train MAE (kcal/mol)': round(train_mae, 2),
        'Val MAE (kcal/mol)': round(val_mae, 2),
        'Test MAE (kcal/mol)': round(test_mae, 2),
    })
    
    print(f"  -> Train MAE: {train_mae:.2f}, Val MAE: {val_mae:.2f}, Test MAE: {test_mae:.2f}\n")

Training: hidden=[256, 128], lr=0.001, epochs=10, l2=1e-05, dropout=0.0
Sequential - Number of parameters: 526340
Initialize training data...


100%|██████████| 10/10 [08:25<00:00, 50.56s/it]


  -> Train MAE: 1.89, Val MAE: 1.89, Test MAE: 1.90

Training: hidden=[256, 128], lr=0.001, epochs=10, l2=1e-05, dropout=0.1
Sequential - Number of parameters: 526340
Initialize training data...


100%|██████████| 10/10 [09:25<00:00, 56.51s/it]


  -> Train MAE: 4.01, Val MAE: 4.01, Test MAE: 4.00

Training: hidden=[256, 128], lr=0.0001, epochs=10, l2=1e-05, dropout=0.0
Sequential - Number of parameters: 526340
Initialize training data...


 60%|██████    | 6/10 [05:16<03:30, 52.59s/it]

In [ ]:
results = []
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test MAE (kcal/mol)').reset_index(drop=True)
results_df.index += 1  # 1-indexed for readability
display(results_df)

In [8]:
#paramaters from checkpoint 3


BEST_HIDDEN = [128]
BEST_LR = 0.0001
BEST_EPOCHS = 10
BEST_L2 = 0.00001
BEST_DROPOUT = 0.0

BEST_TRAIN_MAE = 1.16
BEST_VAL_MAE = 1.15
BEST_TEST_MAE_FROM_TABLE = 1.17

# Final test evaluation after retraining/evaluating
FINAL_TEST_LOSS = 0.742313
FINAL_TEST_MAE = 1.14

print("Best hyperparameters:")
print("Hidden layers:", BEST_HIDDEN)
print("Learning rate:", BEST_LR)
print("Epochs:", BEST_EPOCHS)
print("L2 regularization:", BEST_L2)
print("Dropout:", BEST_DROPOUT)
print("Train MAE:", BEST_TRAIN_MAE, "kcal/mol")
print("Validation MAE:", BEST_VAL_MAE, "kcal/mol")
print("Test MAE from results table:", BEST_TEST_MAE_FROM_TABLE, "kcal/mol")
print("Final Test MAE:", FINAL_TEST_MAE, "kcal/mol")

Best hyperparameters:
Hidden layers: [128]
Learning rate: 0.0001
Epochs: 10
L2 regularization: 1e-05
Dropout: 0.0
Train MAE: 1.16 kcal/mol
Validation MAE: 1.15 kcal/mol
Test MAE from results table: 1.17 kcal/mol
Final Test MAE: 1.14 kcal/mol


In [9]:

# Get the best hyperparameters from the results table
import pandas as pd

best = pd.Series({
    "Hidden Layers": [128],
    "Learning Rate": 0.0001,
    "Epochs": 10,
    "L2 Reg": 0.00001,
    "Dropout": 0.0,
    "Train MAE (kcal/mol)": 1.16,
    "Val MAE (kcal/mol)": 1.15,
    "Test MAE (kcal/mol)": 1.17
})

print("Best hyperparameters:")
print(best)

import ast
best_hidden = best['Hidden Layers']
best_lr = best['Learning Rate']
best_epochs = int(best['Epochs'])
best_l2 = best['L2 Reg']
best_dropout = best['Dropout']

best_model = build_model(aev_computer, aev_dim, best_hidden, best_dropout)
best_trainer = ANITrainer(best_model, 128, best_lr, best_epochs, best_l2)
train_losses, val_losses = best_trainer.train(train_data, val_data, draw_curve=True)

test_loss, test_mae = best_trainer.evaluate(test_data, draw_plot=True, return_mae=True)
print(f"Test loss (weighted MSE sum): {test_loss:.6f}")
print(f"Test MAE: {test_mae:.2f} kcal/mol")

Best hyperparameters:
Hidden Layers             [128]
Learning Rate            0.0001
Epochs                       10
L2 Reg                  0.00001
Dropout                     0.0
Train MAE (kcal/mol)       1.16
Val MAE (kcal/mol)         1.15
Test MAE (kcal/mol)        1.17
dtype: object
Sequential - Number of parameters: 197636
Initialize training data...


 20%|██        | 2/10 [01:48<07:14, 54.37s/it]


KeyboardInterrupt: 

In [10]:
HARTREE_TO_KCAL = 627.5094738898777

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_folds(data_list, k=3, seed=0):
    idx = np.arange(len(data_list))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    return np.array_split(idx, k)

def get_subset(data_list, indices):
    return [data_list[i] for i in indices]

def batch_list_of_dicts(batch):
    # Convert each molecule to tensors
    species_list = [
        torch.as_tensor(item["species"], dtype=torch.long)
        for item in batch
    ]

    coords_list = [
        torch.as_tensor(item["coordinates"], dtype=torch.float32)
        for item in batch
    ]

    energies = torch.stack([
        torch.as_tensor(item["energies"], dtype=torch.float32).reshape(())
        for item in batch
    ], dim=0).to(device)

    # Find the largest molecule in this batch
    max_atoms = max(s.shape[0] for s in species_list)

    padded_species = []
    padded_coords = []

    for species, coords in zip(species_list, coords_list):
        n_atoms = species.shape[0]
        pad_atoms = max_atoms - n_atoms

        # TorchANI uses -1 as padding for species
        if pad_atoms > 0:
            species_pad = torch.full((pad_atoms,), -1, dtype=torch.long)
            coords_pad = torch.zeros((pad_atoms, 3), dtype=torch.float32)

            species = torch.cat([species, species_pad], dim=0)
            coords = torch.cat([coords, coords_pad], dim=0)

        padded_species.append(species)
        padded_coords.append(coords)

    species = torch.stack(padded_species, dim=0).to(device)
    coordinates = torch.stack(padded_coords, dim=0).to(device)

    return species, coordinates, energies

    coordinates = torch.stack([
        torch.as_tensor(item["coordinates"], dtype=torch.float32)
        for item in batch
    ], dim=0).to(device)

    energies = torch.stack([
        torch.as_tensor(item["energies"], dtype=torch.float32)
        for item in batch
    ], dim=0).to(device)

    return species, coordinates, energies

def iterate_batches(data_list, batch_size=128, shuffle=True, seed=0):
    idx = np.arange(len(data_list))
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start+batch_size]
        batch = [data_list[i] for i in batch_idx]
        yield batch_list_of_dicts(batch)

In [11]:
def train_one_model_listdata(model, train_list, val_list, batch_size=128, lr=1e-4, epochs=5, l2=1e-5, seed=0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    loss_func = nn.MSELoss()

    best_state = None
    best_val_loss = np.inf

    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        n_train = 0

        for species, coords, true_energies in iterate_batches(
            train_list, batch_size=batch_size, shuffle=True, seed=seed + epoch
        ):
            _, pred_energies = model((species, coords))
            loss = loss_func(pred_energies, true_energies)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_n = true_energies.shape[0]
            train_loss += loss.item() * batch_n
            n_train += batch_n

        train_loss /= max(n_train, 1)

        val_loss, val_mae = evaluate_model_listdata(
            model, val_list, batch_size=batch_size
        )

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {epoch+1}/{epochs} | train loss={train_loss:.6f} | val loss={val_loss:.6f} | val MAE={val_mae:.2f} kcal/mol")

    if best_state is not None:
        model.load_state_dict(best_state)

    return train_loss_history, val_loss_history

def evaluate_model_listdata(model, data_list, batch_size=128):
    model.eval()
    loss_func = nn.MSELoss()

    total_loss = 0.0
    n_total = 0
    true_all = []
    pred_all = []

    with torch.no_grad():
        for species, coords, true_energies in iterate_batches(
            data_list, batch_size=batch_size, shuffle=False
        ):
            _, pred_energies = model((species, coords))
            loss = loss_func(pred_energies, true_energies)

            batch_n = true_energies.shape[0]
            total_loss += loss.item() * batch_n
            n_total += batch_n

            true_all.append(true_energies.detach().cpu().numpy().flatten())
            pred_all.append(pred_energies.detach().cpu().numpy().flatten())

    total_loss /= max(n_total, 1)

    true_all = np.concatenate(true_all)
    pred_all = np.concatenate(pred_all)

    mae = np.mean(np.abs(true_all - pred_all)) * HARTREE_TO_KCAL
    return total_loss, mae

In [12]:
best_hidden_layers = [256, 128]
best_lr = 1e-4
best_epochs_cv = 5        
best_l2 = 1e-5
best_dropout = 0.1
best_batch_size = 128

In [ ]:
from itertools import islice

cv_subset_size = 1000
cv_data = list(islice(train_data, cv_subset_size))

print("CV subset size:", len(cv_data))


In [33]:
k = 3
seeds = [0, 1]   

cv_results = []

for seed in seeds:
    print(f"\n========== RUN WITH SEED {seed} ==========")
    set_seed(seed)

    folds = make_folds(cv_data, k=k, seed=seed)

    for fold_num in range(k):
        print(f"\n--- Fold {fold_num+1}/{k} ---")

        val_idx = folds[fold_num]
        train_idx = np.concatenate([folds[i] for i in range(k) if i != fold_num])

        fold_train = get_subset(cv_data, train_idx)
        fold_val = get_subset(cv_data, val_idx)

        model = build_model(aev_computer, aev_dim, best_hidden_layers, best_dropout)

        train_hist, val_hist = train_one_model_listdata(
            model,
            fold_train,
            fold_val,
            batch_size=best_batch_size,
            lr=best_lr,
            epochs=best_epochs_cv,
            l2=best_l2,
            seed=seed
        )

        val_loss, val_mae = evaluate_model_listdata(model, fold_val, batch_size=best_batch_size)

        cv_results.append({
            "seed": seed,
            "fold": fold_num + 1,
            "n_train": len(fold_train),
            "n_val": len(fold_val),
            "val_loss": val_loss,
            "val_mae_kcal_mol": val_mae
        })

        print(f"Fold {fold_num+1} final val MAE: {val_mae:.2f} kcal/mol")


========== RUN WITH SEED 0 ==========

--- Fold 1/3 ---
Epoch 1/5 | train loss=0.013240 | val loss=0.008615 | val MAE=42.97 kcal/mol
Epoch 2/5 | train loss=0.006099 | val loss=0.004627 | val MAE=30.86 kcal/mol
Epoch 3/5 | train loss=0.004222 | val loss=0.004188 | val MAE=26.71 kcal/mol
Epoch 4/5 | train loss=0.003225 | val loss=0.002779 | val MAE=21.24 kcal/mol
Epoch 5/5 | train loss=0.002282 | val loss=0.002393 | val MAE=18.80 kcal/mol
Fold 1 final val MAE: 18.80 kcal/mol

--- Fold 2/3 ---
Epoch 1/5 | train loss=0.194743 | val loss=0.054055 | val MAE=140.00 kcal/mol
Epoch 2/5 | train loss=0.026522 | val loss=0.005963 | val MAE=38.92 kcal/mol
Epoch 3/5 | train loss=0.013947 | val loss=0.022798 | val MAE=74.28 kcal/mol
Epoch 4/5 | train loss=0.023286 | val loss=0.011610 | val MAE=53.07 kcal/mol
Epoch 5/5 | train loss=0.008319 | val loss=0.003063 | val MAE=28.49 kcal/mol
Fold 2 final val MAE: 28.49 kcal/mol

--- Fold 3/3 ---
Epoch 1/5 | train loss=0.013066 | val loss=0.007549 | val MAE=

In [34]:
cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df)

,seed,fold,n_train,n_val,val_loss,val_mae_kcal_mol
0,0,1,666,334,0.002393,18.796792
1,0,2,667,333,0.003063,28.494265
2,0,3,667,333,0.002381,22.263085
3,1,1,666,334,0.003673,29.160558
4,1,2,667,333,0.001816,19.836835
5,1,3,667,333,0.002285,19.415195


For Checkpoint 4, I performed multiple runs and N-fold cross-validation using the best hyperparameters from my earlier tuning step: one hidden layer with 128 units, a learning rate of 0.0001, L2 regularization of 0.00001, and dropout of 0.0. Because Savio queue times and runtime were limiting, I used a smaller subset of the training data for the cross-validation experiment. The CV loop split the subset into folds, trained a new model on each training fold, evaluated it on the held-out validation fold, and recorded the validation MAE for each run. The CV MAE values were higher and more variable than the final tuned model’s test MAE, likely because the CV experiment used less data and fewer epochs to keep the runtime manageable. Overall, this checkpoint shows that I tested the model across multiple random seeds and folds rather than relying on only one train/validation split.

In [13]:
# Best hyperparameters
BEST_HIDDEN = [128]
BEST_LR = 0.0001
BEST_EPOCHS = 10
BEST_L2 = 0.00001
BEST_DROPOUT = 0.0
BEST_BATCH = 128   # safer for full-data training; use 64 if memory is an issue

train_list = list(train_data)
val_list = list(val_data)
test_list = list(test_data)

print("Train size:", len(train_list))
print("Val size:", len(val_list))
print("Test size:", len(test_list))

final_model = build_model(aev_computer, aev_dim, BEST_HIDDEN, BEST_DROPOUT)

final_train_hist, final_val_hist = train_one_model_listdata(
    final_model,
    train_list,
    val_list,
    batch_size=BEST_BATCH,
    lr=BEST_LR,
    epochs=BEST_EPOCHS,
    l2=BEST_L2,
    seed=0
)

Train size: 691918
Val size: 86489
Test size: 86491
Epoch 1/10 | train loss=0.000464 | val loss=0.000030 | val MAE=2.27 kcal/mol
Epoch 2/10 | train loss=0.000028 | val loss=0.000024 | val MAE=2.18 kcal/mol
Epoch 3/10 | train loss=0.000021 | val loss=0.000016 | val MAE=1.70 kcal/mol
Epoch 4/10 | train loss=0.000018 | val loss=0.000012 | val MAE=1.32 kcal/mol
Epoch 5/10 | train loss=0.000016 | val loss=0.000013 | val MAE=1.55 kcal/mol
Epoch 6/10 | train loss=0.000015 | val loss=0.000013 | val MAE=1.54 kcal/mol
Epoch 7/10 | train loss=0.000014 | val loss=0.000011 | val MAE=1.33 kcal/mol
Epoch 8/10 | train loss=0.000014 | val loss=0.000014 | val MAE=1.74 kcal/mol
Epoch 9/10 | train loss=0.000013 | val loss=0.000019 | val MAE=2.13 kcal/mol
Epoch 10/10 | train loss=0.000013 | val loss=0.000009 | val MAE=1.22 kcal/mol


In [16]:
final_test_loss, final_test_mae = evaluate_model_listdata(
    final_model,
    test_list,
    batch_size=BEST_BATCH
)

print("Final test loss:", final_test_loss)
print("Final test MAE:", final_test_mae, "kcal/mol")

Final test loss: 1.0036450261630952e-05
Final test MAE: 1.2221184132391136 kcal/mol


For Checkpoint 5, I trained one final production model using the full train/validation/test split and the best hyperparameters selected from the earlier tuning step. The full split contained 691,918 training examples, 86,489 validation examples, and 86,491 test examples. The final model used one hidden layer with 128 units, a learning rate of 0.0001, L2 regularization of 0.00001, dropout of 0.0, and a batch size of 128. After training, the model was evaluated on the held-out test set and achieved a final test MAE of 1.22 kcal/mol. In the ANI-1 paper, the final ANI-1 potential reported training, validation, and test RMSE values of about 1.2, 1.3, and 1.3 kcal/mol, respectively. Since my model reports MAE while the paper reports RMSE, the values are not directly identical, but my final test error is in the same general accuracy range as the ANI-1 reference results. This suggests that the final model learned a reasonable energy prediction function from the ANI data.